# Exploring Embeddings with Keras

In [2]:
import sys

sys.executable

'/home/thomas/Python/virtualenvs/uj_embeddings/bin/python'

## Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances

In [3]:
import keras
from keras import layers
from keras import ops

import math
import numpy as np
import pandas as pd
#from tensorflow import data as tf_data
#import matplotlib.pyplot as plt
from functools import partial

2024-06-25 22:05:24.232237: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-25 22:05:33.261312: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-25 22:05:39.087371: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-25 22:05:51.532833: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [ ]:
CSV_HEADER = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "gender",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income_bracket",
]

train_data_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
)
train_data = pd.read_csv(train_data_url, header=None, names=CSV_HEADER)

test_data_url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
)
test_data = pd.read_csv(test_data_url, header=None, names=CSV_HEADER)

print(f"Train dataset shape: {train_data.shape}")
print(f"Test dataset shape: {test_data.shape}")

In [ ]:
test_data = test_data[1:]
test_data.income_bracket = test_data.income_bracket.apply(
    lambda value: value.replace(".", "")
)

In [ ]:
train_data_file = "train_data.csv"
test_data_file = "test_data.csv"

train_data.to_csv(train_data_file, index=False, header=False)
test_data.to_csv(test_data_file, index=False, header=False)


In [ ]:
# A list of the numerical feature names.
NUMERIC_FEATURE_NAMES = [
    "age",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
]
# A dictionary of the categorical features and their vocabulary.
CATEGORICAL_FEATURES_WITH_VOCABULARY = {
    "workclass": sorted(list(train_data["workclass"].unique())),
    "education": sorted(list(train_data["education"].unique())),
    "marital_status": sorted(list(train_data["marital_status"].unique())),
    "occupation": sorted(list(train_data["occupation"].unique())),
    "relationship": sorted(list(train_data["relationship"].unique())),
    "race": sorted(list(train_data["race"].unique())),
    "gender": sorted(list(train_data["gender"].unique())),
    "native_country": sorted(list(train_data["native_country"].unique())),
}
# Name of the column to be used as instances weight.
WEIGHT_COLUMN_NAME = "fnlwgt"
# A list of the categorical feature names.
CATEGORICAL_FEATURE_NAMES = list(CATEGORICAL_FEATURES_WITH_VOCABULARY.keys())
# A list of all the input features.
FEATURE_NAMES = NUMERIC_FEATURE_NAMES + CATEGORICAL_FEATURE_NAMES
# A list of column default values for each feature.
COLUMN_DEFAULTS = [
    [0.0] if feature_name in NUMERIC_FEATURE_NAMES + [WEIGHT_COLUMN_NAME] else ["NA"]
    for feature_name in CSV_HEADER
]
# The name of the target feature.
TARGET_FEATURE_NAME = "income_bracket"
# A list of the labels of the target features.
TARGET_LABELS = [" <=50K", " >50K"]


In [ ]:
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
DROPOUT_RATE = 0.2
BATCH_SIZE = 265
NUM_EPOCHS = 15

NUM_TRANSFORMER_BLOCKS = 3  # Number of transformer blocks.
NUM_HEADS = 4  # Number of attention heads.
EMBEDDING_DIMS = 16  # Embedding dimensions of the categorical features.
MLP_HIDDEN_UNITS_FACTORS = [
    2,
    1,
]  # MLP hidden layer units, as factors of the number of inputs.
NUM_MLP_BLOCKS = 2  # Number of MLP blocks in the baseline model.


In [ ]:
target_label_lookup = layers.StringLookup(
    vocabulary=TARGET_LABELS, mask_token=None, num_oov_indices=0
)


def prepare_example(features, target):
    target_index = target_label_lookup(target)
    weights = features.pop(WEIGHT_COLUMN_NAME)
    return features, target_index, weights


lookup_dict = {}
for feature_name in CATEGORICAL_FEATURE_NAMES:
    vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]
    # Create a lookup to convert a string values to an integer indices.
    # Since we are not using a mask token, nor expecting any out of vocabulary
    # (oov) token, we set mask_token to None and num_oov_indices to 0.
    lookup = layers.StringLookup(
        vocabulary=vocabulary, mask_token=None, num_oov_indices=0
    )
    lookup_dict[feature_name] = lookup


def encode_categorical(batch_x, batch_y, weights):
    for feature_name in CATEGORICAL_FEATURE_NAMES:
        batch_x[feature_name] = lookup_dict[feature_name](batch_x[feature_name])

    return batch_x, batch_y, weights


def get_dataset_from_csv(csv_file_path, batch_size=128, shuffle=False):
    dataset = (
        tf_data.experimental.make_csv_dataset(
            csv_file_path,
            batch_size=batch_size,
            column_names=CSV_HEADER,
            column_defaults=COLUMN_DEFAULTS,
            label_name=TARGET_FEATURE_NAME,
            num_epochs=1,
            header=False,
            na_value="?",
            shuffle=shuffle,
        )
        .map(prepare_example, num_parallel_calls=tf_data.AUTOTUNE, deterministic=False)
        .map(encode_categorical)
    )
    return dataset.cache()


In [ ]:
def run_experiment(
    model,
    train_data_file,
    test_data_file,
    num_epochs,
    learning_rate,
    weight_decay,
    batch_size,
):
    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name="accuracy")],
    )

    train_dataset = get_dataset_from_csv(train_data_file, batch_size, shuffle=True)
    validation_dataset = get_dataset_from_csv(test_data_file, batch_size)

    print("Start training the model...")
    history = model.fit(
        train_dataset, epochs=num_epochs, validation_data=validation_dataset
    )
    print("Model training finished")

    _, accuracy = model.evaluate(validation_dataset, verbose=0)

    print(f"Validation accuracy: {round(accuracy * 100, 2)}%")

    return history

In [ ]:
def create_model_inputs():
    inputs = {}
    for feature_name in FEATURE_NAMES:
        if feature_name in NUMERIC_FEATURE_NAMES:
            inputs[feature_name] = layers.Input(
                name=feature_name, shape=(), dtype="float32"
            )
        else:
            inputs[feature_name] = layers.Input(
                name=feature_name, shape=(), dtype="int32"
            )
    return inputs


In [ ]:
def encode_inputs(inputs, embedding_dims):
    encoded_categorical_feature_list = []
    numerical_feature_list = []

    for feature_name in inputs:
        if feature_name in CATEGORICAL_FEATURE_NAMES:
            vocabulary = CATEGORICAL_FEATURES_WITH_VOCABULARY[feature_name]
            # Create a lookup to convert a string values to an integer indices.
            # Since we are not using a mask token, nor expecting any out of vocabulary
            # (oov) token, we set mask_token to None and num_oov_indices to 0.

            # Convert the string input values into integer indices.

            # Create an embedding layer with the specified dimensions.
            embedding = layers.Embedding(
                input_dim=len(vocabulary), output_dim=embedding_dims
            )

            # Convert the index values to embedding representations.
            encoded_categorical_feature = embedding(inputs[feature_name])
            encoded_categorical_feature_list.append(encoded_categorical_feature)

        else:
            # Use the numerical features as-is.
            numerical_feature = ops.expand_dims(inputs[feature_name], -1)
            numerical_feature_list.append(numerical_feature)

    return encoded_categorical_feature_list, numerical_feature_list


In [ ]:
def create_mlp(hidden_units, dropout_rate, activation, normalization_layer, name=None):
    mlp_layers = []
    for units in hidden_units:
        mlp_layers.append(normalization_layer())
        mlp_layers.append(layers.Dense(units, activation=activation))
        mlp_layers.append(layers.Dropout(dropout_rate))

    return keras.Sequential(mlp_layers, name=name)


In [ ]:
def create_tabtransformer_classifier(
    num_transformer_blocks,
    num_heads,
    embedding_dims,
    mlp_hidden_units_factors,
    dropout_rate,
    use_column_embedding=False,
):
    # Create model inputs.
    inputs = create_model_inputs()
    # encode features.
    encoded_categorical_feature_list, numerical_feature_list = encode_inputs(
        inputs, embedding_dims
    )
    # Stack categorical feature embeddings for the Tansformer.
    encoded_categorical_features = ops.stack(encoded_categorical_feature_list, axis=1)
    # Concatenate numerical features.
    numerical_features = layers.concatenate(numerical_feature_list)

    # Add column embedding to categorical feature embeddings.
    if use_column_embedding:
        num_columns = encoded_categorical_features.shape[1]
        column_embedding = layers.Embedding(
            input_dim=num_columns, output_dim=embedding_dims
        )
        column_indices = ops.arange(start=0, stop=num_columns, step=1)
        encoded_categorical_features = encoded_categorical_features + column_embedding(
            column_indices
        )

    # Create multiple layers of the Transformer block.
    for block_idx in range(num_transformer_blocks):
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embedding_dims,
            dropout=dropout_rate,
            name=f"multihead_attention_{block_idx}",
        )(encoded_categorical_features, encoded_categorical_features)
        # Skip connection 1.
        x = layers.Add(name=f"skip_connection1_{block_idx}")(
            [attention_output, encoded_categorical_features]
        )
        # Layer normalization 1.
        x = layers.LayerNormalization(name=f"layer_norm1_{block_idx}", epsilon=1e-6)(x)
        # Feedforward.
        feedforward_output = create_mlp(
            hidden_units=[embedding_dims],
            dropout_rate=dropout_rate,
            activation=keras.activations.gelu,
            normalization_layer=partial(
                layers.LayerNormalization, epsilon=1e-6
            ),  # using partial to provide keyword arguments before initialization
            name=f"feedforward_{block_idx}",
        )(x)
        # Skip connection 2.
        x = layers.Add(name=f"skip_connection2_{block_idx}")([feedforward_output, x])
        # Layer normalization 2.
        encoded_categorical_features = layers.LayerNormalization(
            name=f"layer_norm2_{block_idx}", epsilon=1e-6
        )(x)

    # Flatten the "contextualized" embeddings of the categorical features.
    categorical_features = layers.Flatten()(encoded_categorical_features)
    # Apply layer normalization to the numerical features.
    numerical_features = layers.LayerNormalization(epsilon=1e-6)(numerical_features)
    # Prepare the input for the final MLP block.
    features = layers.concatenate([categorical_features, numerical_features])

    # Compute MLP hidden_units.
    mlp_hidden_units = [
        factor * features.shape[-1] for factor in mlp_hidden_units_factors
    ]
    # Create final MLP.
    features = create_mlp(
        hidden_units=mlp_hidden_units,
        dropout_rate=dropout_rate,
        activation=keras.activations.selu,
        normalization_layer=layers.BatchNormalization,
        name="MLP",
    )(features)

    # Add a sigmoid as a binary classifer.
    outputs = layers.Dense(units=1, activation="sigmoid", name="sigmoid")(features)
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


tabtransformer_model = create_tabtransformer_classifier(
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    num_heads=NUM_HEADS,
    embedding_dims=EMBEDDING_DIMS,
    mlp_hidden_units_factors=MLP_HIDDEN_UNITS_FACTORS,
    dropout_rate=DROPOUT_RATE,
)

print("Total model weights:", tabtransformer_model.count_params())
keras.utils.plot_model(tabtransformer_model, show_shapes=True, rankdir="LR")

In [ ]:
history = run_experiment(
    model=tabtransformer_model,
    train_data_file=train_data_file,
    test_data_file=test_data_file,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
)

In [2]:
['cpf9_espec',
 'id_eq3_espec',
 'cnpj_espec',
 'status_espec',
 'ano_mes_dia',
 'qtde_sub',
 'ult_ref_reneg',
 'flag pjampliada', 
 'qtd contratos',
 'qtd cntrt novo',
 'qtd cntrt renov int',
 'qtd_cntrt_renov merc',
 'qtd cntrts_en aberto',
 'qtd_cntrt_reneg',
 'qtd_fpd_reneg',
 'qtd cntrts pivo', 
 'qtd_cntrts_app', 
 'avg_idade_clientes',
 'qtd_cntrts_sld_pos_bc',
 'avg_mob_quit_cntrt', 
 'avg plano_cntrt', 
 'rz_vlr_entregue_c0c1',
 'rz_panis_pa',
 'rz_entregue_pa',
 'rz_entregue_panis',
 'avg_saldo_em_aberto_parc',
 'pct_cntrt_app',
 'pct_cntrts_reneg']

['cpf9_espec',
 'id_eq3_espec',
 'cnpj_espec',
 'status_espec',
 'ano_mes_dia',
 'qtde_sub',
 'ult_ref_reneg',
 'flag pjampliada',
 'qtd contratos',
 'qtd cntrt novo',
 'qtd cntrt renov int',
 'qtd_cntrt_renov merc',
 'qtd cntrts_en aberto',
 'qtd_cntrt_reneg',
 'qtd_fpd_reneg',
 'qtd cntrts pivo',
 'qtd_cntrts_app',
 'avg_idade_clientes',
 'qtd_cntrts_sld_pos_bc',
 'avg_mob_quit_cntrt',
 'avg plano_cntrt',
 'rz_vlr_entregue_c0c1',
 'rz_panis_pa',
 'rz_entregue_pa',
 'rz_entregue_panis',
 'avg saldo_em_aberto_parc',
 'pct_cntrt_app',
 'pct_cntrts_reneg']

- CONSIGNADO
    - Mix de propostas      -> Propostas por GH (2305) (Todos SPI cujas empresas possuem convênio com o banco (nrregra in ('OP','EP')). Propostas APROVADAS = quem possui PA no público acima)
    - Mix de contratação    -> Contratos por GH (2306)
    - Mix de inadimplência  -> Bad por GH
    * No mix de contratação, devemos utilizar o mesmo público de propostas como denominador, ou levamos em conta somente a referência?
    * No mix de contratação, utilizamos somente os contratos internos?
    * No mix de inadimplência, 100% interno? ou 50% 50% com mercado? 
      
- IMOB
    - Mix de propostas      -> Propostas por GH (2303)
    - Mix de contratação    -> Contratos por GH (2304)
    - Mix de inadimplência  -> Bad por GH
 


- PROPOSTAS
  - CONSIG: Cruzar as bases de produto (que identificam o público SPI de empresas conveniadas) com a base do modelo (GH)
  - IMOB: Cruzar base de propostas do produto (SAS, csv) com a base do modelo (GH)
 
- APROVAÇÃO
  - CONSIG: Cruzar a base de propostas acima (já marcada com GH do modelo) com a de produtos (que marcam o PA)
  - IMOB: A base de propostas já traz info de decisao
 
- CONTRATAÇÂO
  - CONSIG: Cruzar a base de aprovação acima (com GH e decisão de proposta) com a base do FOUCAULT CONTRATAÇÃO
  - IMOB: Cruzar a base de propostas acima (com GH e decisão de proposta) com a base do FOUCAULT CONTRATAÇÃO
 
- INADIMPLENCIA
  - CONSIG: Cruzar as bases de contratação acima (com GH, decisão de proposta e info de contratação) com as bases do FOUCAULT PD ITAU e PD MERC
  - IMOB: Cruzar as bases de contratação acima (com GH, decisão de proposta e info de contratação) com as bases do FOUCAULT PD ITAU e PD MERC

- MICROCRÉDITO
  - [ ] Revisar o target (ever30em3 - se tornou ddd em algum momento nos próximos 3 meses) 
  - [ ] Avaliar gini somente no público hhh
  - [ ] Variáveis PJ - Sinal Vital PJ
  - [ ] Construir matrizes de Grupos de Risco de acordo com a política
    - [ ] Caclular inadimplência (gini) com a classificação deles VS com a nova classificação proposta
  - [ ] Gerar visão de threshold (Karina - se definir escore de corte em X vou colocar Y% a mais/menos o que traz um resultado de Z% a mais/menos)
  - [ ] Explicabilidade para os diferentes grupos (todos, vs hhh)

In [ ]:
politica_micro = ['Novos','Renov_Interna','Renov_Mercado']

- Acrescentar ao processo, join com a tabela de contratos
  - [ ] Cruzamento com modelo eleito PF (Polaris/VPRO)
  - [ ] Cruzamento com Meridian Microcredito (Capivara)
 
- GRUPO POLARIS
  - VP1: A1 - C3
  - VP2: C4 - D1
  - VP3: D2 - D4
  - VP4: E1 - E2
  - VP5: E3 - E4
  - VP6: F1
  - VP7: F2
  - VP8: F3 - f4

In [5]:
grupo_polaris = {
    'VP1': ['A1','A2','A3','B1','B2','B3','B4','C1','C2','C3'],
    'VP2': ['C4','D1'],
    'VP3': ['D2','D3','D4'],
    'VP4': ['E1','E2'],
    'VP5': ['E3','E4'],
    'VP6': ['F1'],
    'VP7': ['F2'],
    'VP8': ['F3','F4']
}

grupo_polaris
# conditions = [full_table.rating_polaris.isin(v) for k,v in grupo_polaris.items()]
# choices = [k for k,_ in grupo_polaris.items()]

# full_table['grupo_polaris'] == np.select(conditions, choices)

{'VP1': ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'B4', 'C1', 'C2', 'C3'],
 'VP2': ['C4', 'D1'],
 'VP3': ['D2', 'D3', 'D4'],
 'VP4': ['E1', 'E2'],
 'VP5': ['E3', 'E4'],
 'VP6': ['F1'],
 'VP7': ['F2'],
 'VP8': ['F3', 'F4']}

In [16]:
grupo_polaris_hexagono = np.array([
    ['VPH1']*5,
    ['VPH1']*3 + ['VPH2']*2,
    ['VPH1']*2 + ['VPH2']*1 + ['VPH3']*2,
    ['VPH2']*2 + ['VPH3']*1 + ['VPH4']*2,
    ['VPH3']*2 + ['VPH4']*2 + ['VPH5']*1,
    ['VPH4']*2 + ['VPH5']*3,
    ['VPH5']*3 + ['VPH7']*2,
    ['VPH6']*2 + ['VPH7']*1 + ['VPH8']*2
])

df_vph = pd.DataFrame(grupo_polaris_hexagono, index=list(grupo_polaris.keys()), columns=[x for x in range(1,6)])
df_vph

,1,2,3,4,5
VP1,VPH1,VPH1,VPH1,VPH1,VPH1
VP2,VPH1,VPH1,VPH1,VPH2,VPH2
VP3,VPH1,VPH1,VPH2,VPH3,VPH3
VP4,VPH2,VPH2,VPH3,VPH4,VPH4
VP5,VPH3,VPH3,VPH4,VPH4,VPH5
VP6,VPH4,VPH4,VPH5,VPH5,VPH5
VP7,VPH5,VPH5,VPH5,VPH7,VPH7
VP8,VPH6,VPH6,VPH7,VPH8,VPH8


In [12]:
df_vph.at['VP8',1]

'VPH6'

In [31]:
grupo_risco_meridian = np.array([
    ['GRM1']*4 + ['GRM2']*2 + ['GRM4'] + ['GRM5']*2,
    ['GRM1']*1 + ['GRM2']*3 + ['GRM3']*2 + ['GRM4']*1 + ['GRM5']*2,
    ['GRM2']*3 + ['GRM3']*2 + ['GRM4']*2 + ['GRM5']*2,
    ['GRM3']*4 + ['GRM4']*3 + ['GRM5']*2,
    ['GRM4']*5 + ['GRM5']*4,
    ['GRM5']*5 + ['GRM6']*4,
    ['GRM6']*7 + ['GRM7']*2,
    ['GRM6']*5 + ['GRM7']*4
    
    
])

df_grm = pd.DataFrame(grupo_risco_meridian, index=np.sort(pd.unique(df_vph.values.ravel())), columns=[x for x in range(1,10)])
df_grm

,1,2,3,4,5,6,7,8,9
VPH1,GRM1,GRM1,GRM1,GRM1,GRM2,GRM2,GRM4,GRM5,GRM5
VPH2,GRM1,GRM2,GRM2,GRM2,GRM3,GRM3,GRM4,GRM5,GRM5
VPH3,GRM2,GRM2,GRM2,GRM3,GRM3,GRM4,GRM4,GRM5,GRM5
VPH4,GRM3,GRM3,GRM3,GRM3,GRM4,GRM4,GRM4,GRM5,GRM5
VPH5,GRM4,GRM4,GRM4,GRM4,GRM4,GRM5,GRM5,GRM5,GRM5
VPH6,GRM5,GRM5,GRM5,GRM5,GRM5,GRM6,GRM6,GRM6,GRM6
VPH7,GRM6,GRM6,GRM6,GRM6,GRM6,GRM6,GRM6,GRM7,GRM7
VPH8,GRM6,GRM6,GRM6,GRM6,GRM6,GRM7,GRM7,GRM7,GRM7


In [32]:
grupo_risco_renovacao = np.array([
    ['GRR1']*2 + ['GRR2']*7,
    ['GRR1']*1 + ['GRR2']*1 + ['GRR3']*4 + ['GRR4']*3,
    ['GRR2']*2 + ['GRR3']*3 + ['GRR4']*4,
    ['GRR3']*2 + ['GRR4']*3 + ['GRR5']*1 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*4 + ['GRR6']*3,
    ['GRR4']*2 + ['GRR5']*2 + ['GRR6']*5    
])

df_grr = pd.DataFrame(grupo_risco_renovacao, index=list(grupo_polaris.keys()), columns=[x for x in range(1,10)])
df_grr

,1,2,3,4,5,6,7,8,9
VP1,GRR1,GRR1,GRR2,GRR2,GRR2,GRR2,GRR2,GRR2,GRR2
VP2,GRR1,GRR2,GRR3,GRR3,GRR3,GRR3,GRR4,GRR4,GRR4
VP3,GRR2,GRR2,GRR3,GRR3,GRR3,GRR4,GRR4,GRR4,GRR4
VP4,GRR3,GRR3,GRR4,GRR4,GRR4,GRR5,GRR6,GRR6,GRR6
VP5,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP6,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP7,GRR4,GRR4,GRR5,GRR5,GRR5,GRR5,GRR6,GRR6,GRR6
VP8,GRR4,GRR4,GRR5,GRR5,GRR6,GRR6,GRR6,GRR6,GRR6


MATRIZES DE APROVAÇAO

In [62]:
class_espec = ['aam','aaa','bbb','ccc','ddd','hhh','ppp']

NOVOS

In [63]:
n_class_espec = len(class_espec)
n_grps_grm = len(pd.unique(df_grm.values.ravel()))

mat_aprov_novos = pd.DataFrame(np.ones((n_grps_grm,n_class_espec)), index=np.sort(pd.unique(df_grm.values.ravel())), columns=class_espec)
novos_neg = [('GRM3','ddd'), 
             ('GRM4','ccc'), ('GRM4','ddd'), 
             ('GRM5','bbb'), ('GRM5','ccc'), ('GRM5','ddd'),
             ('GRM6','aaa'), ('GRM6','bbb'), ('GRM6','ccc'), ('GRM6','ddd'), ('GRM6','hhh'), ('GRM6','ppp'),
             ('GRM7','aam'), ('GRM7','aaa'), ('GRM7','bbb'), ('GRM7','ccc'), ('GRM7','ddd'), ('GRM7','hhh'), ('GRM7','ppp')]

for grm, clasf in novos_neg:
    mat_aprov_novos.loc[grm,clasf] = 0

mat_aprov_novos

,aam,aaa,bbb,ccc,ddd,hhh,ppp
GRM1,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GRM2,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GRM3,1.0,1.0,1.0,1.0,0.0,1.0,1.0
GRM4,1.0,1.0,1.0,0.0,0.0,1.0,1.0
GRM5,1.0,1.0,0.0,0.0,0.0,1.0,1.0
GRM6,1.0,0.0,0.0,0.0,0.0,0.0,0.0
GRM7,0.0,0.0,0.0,0.0,0.0,0.0,0.0


RENOVACAO INTERNA

In [64]:
pd.unique(df_grr.values.ravel())

array(['GRR1', 'GRR2', 'GRR3', 'GRR4', 'GRR5', 'GRR6'], dtype=object)

In [65]:
n_class_espec, n_grps_grr

(7, 6)

In [66]:
len(class_espec)

7

In [67]:
n_class_espec = len(class_espec)
n_grps_grr = len(pd.unique(df_grr.values.ravel()))

mat_aprov_renov_int = pd.DataFrame(np.ones((n_grps_grr,n_class_espec)), 
                                   index=np.sort(pd.unique(df_grr.values.ravel())), 
                                   columns=class_espec)
renov_int_neg = [('GRR6','ccc'), ('GRR6','ddd')]

for grr, clasf in renov_int_neg:
    mat_aprov_renov_int.loc[grr,clasf] = 0

mat_aprov_renov_int.astype(bool)

,aam,aaa,bbb,ccc,ddd,hhh,ppp
GRR1,True,True,True,True,True,True,True
GRR2,True,True,True,True,True,True,True
GRR3,True,True,True,True,True,True,True
GRR4,True,True,True,True,True,True,True
GRR5,True,True,True,True,True,True,True
GRR6,True,True,True,False,False,True,True


In [55]:
np.ones((n_grps_grr,n_class_espec))

array([[1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.]])

In [ ]:
full_table.politica_micro

In [ ]:
conditions = [arr1 > 2, arr2 < 9]
choices = [arr1, arr2 * 2]

result = np.select(conditions, choices, default=0)